# LangChain / Chroma retrieval


In [ ]:
%pip install -q torch==2.13.0 transformers==5.16.1 datasets==5.0.1 requests \
chromadb langchain langchain-community langchain-openai langchain-chroma openai pandas


In [ ]:
import json, random, re, string, time
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from datasets import Dataset, DatasetDict

SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_REPO="https://github.com/Gokcimen/Home_Appliance_Dataset"
!rm -rf /content/Home_Appliance_Dataset
!git clone -q --depth 1 {DATA_REPO}.git /content/Home_Appliance_Dataset

DATA_ROOT=Path("/content/Home_Appliance_Dataset")

def flatten(path):
    raw=json.loads(Path(path).read_text(encoding="utf-8"))
    rows=[]
    for article in raw["data"]:
        title=article["title"]
        for para in article["paragraphs"]:
            context=para["context"]
            for qa in para["qas"]:
                rows.append({
                    "id":str(qa["id"]),
                    "title":title,
                    "context":context,
                    "question":qa["question"],
                    "answers":{
                        "text":[a["text"] for a in qa["answers"]],
                        "answer_start":[int(a["answer_start"]) for a in qa["answers"]],
                    }
                })
    return rows

raw_datasets=DatasetDict({
    "train":Dataset.from_list(flatten(DATA_ROOT/"train.json")),
    "validation":Dataset.from_list(flatten(DATA_ROOT/"dev.json")),
    "test":Dataset.from_list(flatten(DATA_ROOT/"test.json")),
})

assert len(raw_datasets["train"])==8000
assert len(raw_datasets["validation"])==1000
assert len(raw_datasets["test"])==1000

ids={s:set(raw_datasets[s]["id"]) for s in raw_datasets}
assert not ids["train"]&ids["validation"]
assert not ids["train"]&ids["test"]
assert not ids["validation"]&ids["test"]
all_ids=set().union(*ids.values())
assert len(all_ids)==10000
assert {int(x) for x in all_ids}==set(range(1,10001))

titles={s:set(raw_datasets[s]["title"]) for s in raw_datasets}
assert not titles["train"]&titles["validation"]
assert not titles["train"]&titles["test"]
assert not titles["validation"]&titles["test"]
assert len(set().union(*titles.values()))==1111

print("train",len(raw_datasets["train"]))
print("validation",len(raw_datasets["validation"]))
print("test",len(raw_datasets["test"]))
print("products",len(set().union(*titles.values())))


In [ ]:
CHUNK_SIZE=1000
CHUNK_OVERLAP=0
TOP_K=4
SIMILARITY="cosine"
EMBEDDING_MODEL="text-embedding-ada-002"

train_contexts={}
for row in raw_datasets["train"]:
    train_contexts[row["title"]]=row["context"]

print("train product contexts:",len(train_contexts))


In [ ]:
from langchain.text_splitter import CharacterTextSplitter

splitter=CharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

documents=[]
for title,context in train_contexts.items():
    parts=splitter.create_documents([context],metadatas=[{"title":title,"split":"train"}])
    documents.extend(parts)

print("chunks:",len(documents))


In [ ]:

import os

if os.getenv("OPENAI_API_KEY"):
    from langchain_openai import OpenAIEmbeddings
    from langchain_chroma import Chroma

    embeddings=OpenAIEmbeddings(model=EMBEDDING_MODEL)
    db=Chroma.from_documents(
        documents,
        embedding=embeddings,
        collection_metadata={"hnsw:space":"cosine"},
    )
    retriever=db.as_retriever(search_kwargs={"k":TOP_K})
    print("retriever ready")
else:
    print("OPENAI_API_KEY the index will be created when the required API key is defined.")
